Build a workflow using CrewAI tools to scrape the content from the website and then perform RAG on it. 

Install the crewai-tools and crewai packages using pip:

In [ ]:
# Install CrewAI with tools support
# pip install 'crewai[tools]'

# Or using uv (recommended by CrewAI for dependency management)
# uv add 'crewai[tools]'

In [1]:
# Check package version

import importlib.metadata

# List the distribution package names
packages = ["crewai", "crewai-tools"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


crewai version: 1.15.21
crewai-tools version: 1.15.21


In this example, we’ll use three different tools: 
- ScrapeWebsiteTool to scrape the website, 
- FileWriterTool to write the file, and 
- TXTSearchTool to search the context for RAG. 

## Step 1: Scraping a website
We initialize the ScrapeWebsiteTool. This tool is used to extract content from a website. 

In this case, it’s set to scrape content from Wikipedia's "Artificial Intelligence" page:

In [2]:
from crewai_tools import ScrapeWebsiteTool, FileWriterTool, TXTSearchTool
import requests

# Initialize the tool, potentially passing the session
tool = ScrapeWebsiteTool(website_url='https://en.wikipedia.org/wiki/Artificial_intelligence')  

# Extract the text
text = tool.run()
print(text)

The following text is scraped website content:
Artificial intelligence - Wikipedia
Jump to content
Main menu
Main menu
move to sidebar
hide
Navigation
Main page Contents Current events Random article About Wikipedia Contact us
Contribute
Help Learn to edit Community portal Recent changes Upload file Special pages
Search
Search
Appearance
Donate
Create account
Log in
Personal tools
Donate
Create account
Log in
Contents
move to sidebar
hide
(Top)
1
Goals
Toggle Goals subsection
1.1
Reasoning and problem-solving
1.2
Knowledge representation
1.3
Planning and decision-making
1.4
Learning
1.5
Natural language processing
1.6
Perception
1.7
Social intelligence
1.8
General intelligence
2
Techniques
Toggle Techniques subsection
2.1
Search and optimisation
2.1.1
State space search
2.1.2
Local search
2.2
Logic
2.3
Probabilistic methods for uncertain reasoning
2.4
Classifiers and statistical learning methods
2.5
Artificial neural networks
2.5.1
Deep learning
2.5.2
GPTs and LLMs
2.6
Hardware and sof

## Step2: Write the extracted text to a file
We now use the **FileWriterTool** to save the extracted text into a file named FILENAME

In [3]:
# Initialize the tool
file_writer_tool = FileWriterTool()
FILENAME = '3_webresearch_ai.txt'

# Write content to a file in a specified directory
result = file_writer_tool._run(filename=FILENAME, content = text, directory = '', overwrite=True)
print(result)

Content successfully written to 3_webresearch_ai.txt


## Step 3: Set up the text search tool
We set up another tool to search the contents of the file FILENAME that we just saved. We also set the environment variable for the OpenAI API key.

In [14]:
import os
import getpass
from crewai import LLM

# 1. Prompt securely for the Gemini API Key
gemini_key = getpass.getpass("Enter your Gemini API Key: ")

# 2. Set the appropriate environment variable for Gemini
os.environ["GEMINI_API_KEY"] = gemini_key

# 3. Initialize your CrewAI LLM instance pointing to Gemini
# Popular options include: 'gemini/gemini-2.5-flash' or 'gemini/gemini-2.5-pro'
gemini_llm = LLM(
    model="gemini/gemini-2.5-flash",
    api_key=gemini_key,
    temperature=0.7
)



# Initialize the tool with a specific text file, so the agent can search within the given text file's content
tool = TXTSearchTool(txt=FILENAME)

Enter your Gemini API Key:  ········


## Step 4: Create an agent for the task and execute 

We create a data analyst agent with the role of an educator. This agent's task is to answer the question "What is Natural Language Processing?" based on the text we searched in the file.

In [15]:
from crewai import Agent, Task, Crew

context = tool.run('What is natural language processing?')

data_analyst = Agent(
    role='Educator',
    goal=f'Based on the context provided, answer the question - What is Natural Language Processing? Context - {context}',
    backstory='You are a data expert',
    verbose=True,
    allow_delegation=False,
    tools=[tool],
    llm=gemini_llm,  # <--- Connect Gemini here
)

test_task = Task(
    description="Using the provided context, answer the question: What is Natural Language Processing?",
    tools=[tool],
    agent=data_analyst,
    expected_output='A clear, 2-3 paragraph explanation of Natural Language Processing, covering its \
             definition, key techniques, and common applications.'
)

# Crew with one agent and one task
crew = Crew(
    agents=[data_analyst],
    tasks=[test_task]
)

# output = crew.kickoff() # if using script

# For notebook Use kickoff_async and await the result
output = await crew.kickoff_async()

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Educator                                                                                                │
│                                                                                                                 │
│  Task: Using the provided context, answer the question: What is Natural Language Processing?                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Educator                                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Natural language processing (NLP) is a field that enables computer programs to understand, generate, and       │
│  interact in human languages. This capability allows machines to read, write, and communicate in ways similar   │
│  to humans. NLP addresses a variety of specific problems, including speech recognition, which converts spoken   │
│  language into text; speech synthesis, which generates human-like speech from text; machine translation, for    │
│  converting text or speech from one language to another; information extraction, which identifies structured    │
│  information from unstructured text; information retrieval, for finding relevant documents or information       │
│  based on a query; and question answering, which provides direct answers to questions posed in natural          │
│  language.                                                                                                      │
│                                                                                                                 │
│  Historically, early NLP efforts, drawing from Noam Chomsky's generative grammar and semantic networks, faced   │
│  challenges with word-sense disambiguation, often requiring narrow "micro-worlds" due to the complexities of    │
│  common sense knowledge. Margaret Masterman, a British linguist, emphasized meaning and the role of             │
│  dictionaries and thesauri as foundational for computational language structure. Modern NLP has significantly   │
│  advanced through deep learning techniques. These include word embedding, which represents words as vectors to  │
│  encode their meaning; transformers, a deep learning architecture utilizing an attention mechanism; and other   │
│  sophisticated methods. A notable development in 2019 was the emergence of generative pre-trained transformer   │
│  (GPT) language models, which became capable of generating coherent text. By 2023, these models demonstrated    │
│  human-level performance on various real-world assessments, including the bar exam, SAT, and GRE.               │
│                                                                                                                 │
│  Generative pre-trained transformers (GPTs) are a type of large language model (LLM) that produce text by       │
│  analyzing the semantic relationships between words within sentences. These models undergo extensive            │
│  pre-training on vast datasets of text, often sourced from the internet, where they learn to predict the next   │
│  token (a word, subword, or punctuation mark). Through this process, GPT models acquire a broad understanding   │
│  of the world, enabling them to generate human-like text by iteratively predicting subsequent tokens. Further   │
│  refinement typically involves a training phase, such as reinforcement learning from human feedback (RLHF), to  │
│  enhance their truthfulness, usefulness, and safety. These systems are widely employed in applications like     │
│  chatbots, facilitating natural language interaction for users to ask questions or request tasks. Deep          │
│  learning, as a whole, plays a crucial role in improving the performance of NLP, alongside other areas like     │
│  computer vision and speech recognition.                                                                        │
│                                                        

In [16]:
output.raw

'Natural language processing (NLP) is a field that enables computer programs to understand, generate, and interact in human languages. This capability allows machines to read, write, and communicate in ways similar to humans. NLP addresses a variety of specific problems, including speech recognition, which converts spoken language into text; speech synthesis, which generates human-like speech from text; machine translation, for converting text or speech from one language to another; information extraction, which identifies structured information from unstructured text; information retrieval, for finding relevant documents or information based on a query; and question answering, which provides direct answers to questions posed in natural language.\n\nHistorically, early NLP efforts, drawing from Noam Chomsky\'s generative grammar and semantic networks, faced challenges with word-sense disambiguation, often requiring narrow "micro-worlds" due to the complexities of common sense knowledge